# 31 · Saídas do Google Play de 2025 para 2026

O churn do plano mensal no Google Play foi de 8,4% (2025) para 11,3% (2026), a maior alta entre os
canais de cobrança. Este notebook localiza quando a alta acontece e testa explicações: perfil de
quem assina, preço, falha de GPS do app, suporte e motivo declarado.

Três fatores: ativos (base no início do mês), engajamento antes da saída (`src/engajamento.py`) e
churn (perdas ÷ ativas no início do mês).

Atenção: `sistema` (android/ios) é a situação na exportação, sem data (decisoes.md). Aqui ele é
usado só para descrever, e bate 100% com a loja de quem assina por loja.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")

assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "sistema", "canal_aquisicao", "versao_app", "idade_no_corte", "sexo"])
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "canal_cobranca", "ativa_inicio_mes", "iniciou_no_mes", "perdida_no_mes", "motivo_fim", "n_renovacao_automatica_desligada"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "origem", "gps_ok"])
tickets = pd.read_parquet(P / "tickets_suporte_limpa.parquet")
pesq = pd.read_parquet(P / "pesquisa_saida_limpa.parquet")
pag = pd.read_parquet(P / "pagamentos_limpo.parquet", columns=["user_id", "mes", "status", "metodo", "valor"])

p = painel.merge(usuarios, on="user_id", how="left", validate="many_to_one")
print("painel:", len(painel), "-> depois da junção:", len(p), "| sem sistema:", int(p.sistema.isna().sum()))
p = p[p.mes >= "2025-01-01"].copy()
p["ano"] = p.mes.dt.year
print("painel 2025-2026:", len(p))
print("sistema × canal de cobrança (assinaturas)"); print(pd.crosstab(assin.merge(usuarios, on="user_id").sistema, assin.canal_cobranca))

def churn(d, por):
    g = d.groupby(por).agg(ativas=("ativa_inicio_mes", "sum"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    return g

painel: 327771 -> depois da junção: 327771 | sem sistema: 0
painel 2025-2026: 282573
sistema × canal de cobrança (assinaturas)
canal_cobranca  app_store  google_play  web_cartao  web_pix
sistema                                                    
android                 0        10948        4200     1964
ios                 11170            0        2887     1414


## 1. Onde está a alta: plano, mês e canal

In [2]:
print("churn % por ano, plano e canal"); print(churn(p, ["plano", "canal_cobranca", "ano"])["churn %"].unstack())
print()
m = p[p.plano == "mensal"]
s = churn(m, ["mes", "canal_cobranca"])
print("MENSAL: churn % por mês e canal"); print(s["churn %"].unstack())
print(); print("MENSAL: perdas por mês e canal"); print(s.perdas.unstack().loc["2026-01-01":])
print(); print("MENSAL: ativas no início do mês"); print(s.ativas.unstack().loc["2026-01-01":])
s["churn %"].unstack().to_csv(S / "google_play_churn_mensal_por_canal.csv", encoding="utf-8-sig")

churn % por ano, plano e canal
ano                    2025  2026
plano  canal_cobranca            
anual  app_store        2.6   3.1
       google_play      2.7   3.4
       web_cartao       2.9   3.6
       web_pix          2.8   3.2
mensal app_store        8.6   9.6
       google_play      8.4  11.3
       web_cartao      10.2  12.4
       web_pix          9.1  10.9



MENSAL: churn % por mês e canal
canal_cobranca  app_store  google_play  web_cartao  web_pix
mes                                                        
2025-01-01           10.3         10.6        11.2     10.8
2025-02-01            9.0          8.4        12.4      8.4
2025-03-01            7.7         10.7        10.7      8.9
2025-04-01            9.2          6.6        10.4      9.7
2025-05-01            7.9          9.2         8.5      8.9
2025-06-01            8.6          7.7        10.1      8.0
2025-07-01            9.1          8.4        10.3      8.5
2025-08-01            9.5          9.6        11.4     10.7
2025-09-01            9.1          7.4         7.6      9.1
2025-10-01            8.1          8.0         9.5      8.9
2025-11-01            7.5          8.2        10.4      9.7
2025-12-01            7.6          7.0        10.1      8.0
2026-01-01           10.5         12.0        11.5      9.1
2026-02-01            8.3          9.2        10.9      8.6
2026-03-

In [3]:
# Por sistema do aparelho: o site tem assinantes dos dois sistemas
m = m.assign(onde=m.canal_cobranca.map({"app_store": "loja", "google_play": "loja", "web_cartao": "site", "web_pix": "site"}))
print("MENSAL: churn % por mês, sistema e loja/site")
print(churn(m[m.mes >= "2026-01-01"], ["mes", "sistema", "onde"])["churn %"].unstack(["sistema", "onde"]))
print()
print("MENSAL: churn % por período e sistema")
m["periodo"] = np.select([m.ano == 2025, m.mes <= "2026-03-01", m.mes <= "2026-05-01", m.mes <= "2026-07-01"],
                         ["1. 2025", "2. jan-mar/26", "3. abr-mai/26", "4. jun-jul/26"], "5. ago-set/26")
x = churn(m, ["periodo", "sistema"]); print(x["churn %"].unstack()); print(x.perdas.unstack())
print(); print("por tipo de saída, mensal, % das ativas");
for mot in ["renovacao_desligada", "falha_pagamento"]:
    g = m.assign(k=m.perdida_no_mes & (m.motivo_fim == mot)).groupby(["periodo", "sistema"]).agg(a=("ativa_inicio_mes", "sum"), k=("k", "sum"))
    print(mot); print((g.k / g.a * 100).unstack())

MENSAL: churn % por mês, sistema e loja/site
sistema    android       ios     
onde          loja site loja site
mes                              
2026-01-01    12.0 11.9 10.5  8.7
2026-02-01     9.2 10.6  8.3  9.2
2026-03-01    10.0 12.5 10.7 13.2
2026-04-01    14.7 16.1 12.8 16.1
2026-05-01    13.3 13.6 12.7 13.8
2026-06-01    13.0 15.6  8.5  7.4
2026-07-01    13.7 15.6  8.6  9.9
2026-08-01     9.5  9.8  8.0 11.3
2026-09-01     5.8  7.0  6.3  7.2

MENSAL: churn % por período e sistema
sistema        android  ios
periodo                    
1. 2025            8.9  8.9
2. jan-mar/26     10.8 10.0
3. abr-mai/26     14.3 13.4
4. jun-jul/26     14.1  8.6
5. ago-set/26      7.9  7.7
sistema        android   ios
periodo                     
1. 2025           2986  2627
2. jan-mar/26     1061   903
3. abr-mai/26      930   813
4. jun-jul/26      843   505
5. ago-set/26      443   468

por tipo de saída, mensal, % das ativas


renovacao_desligada


sistema        android  ios
periodo                    
1. 2025            7.9  8.1
2. jan-mar/26      9.8  9.2
3. abr-mai/26     13.2 12.5
4. jun-jul/26     13.2  7.6
5. ago-set/26      6.7  6.7
falha_pagamento
sistema        android  ios
periodo                    
1. 2025            1.0  0.8
2. jan-mar/26      1.0  0.7
3. abr-mai/26      1.1  0.8
4. jun-jul/26      0.9  0.9
5. ago-set/26      1.1  0.9


## 2. Explicações de perfil e preço

In [4]:
print("preço cobrado no mensal (aprovado), por método: valor mais comum por mês")
pg = pag[(pag.status == "aprovado") & pag.valor.isin([34.9, 39.9]) & (pag.mes >= "2026-01-01")]
print((pg.assign(novo=pg.valor == 39.9).groupby(["mes", "metodo"]).novo.mean() * 100).unstack().round(0))
print()
a = assin.merge(usuarios, on="user_id", how="left")
ini = a[(a.plano_atual == "mensal") & (a.inicio >= "2025-01-01")].assign(semestre=lambda d: d.inicio.dt.year.astype(str) + np.where(d.inicio.dt.month <= 6, "-S1", "-S2"))
print("novas assinaturas mensais por semestre e canal"); print(pd.crosstab(ini.semestre, ini.canal_cobranca))
print(); print("% vindas de influenciador, por semestre de início e canal")
print(ini.assign(inf=ini.canal_aquisicao == "influenciador").pivot_table(index="semestre", columns="canal_cobranca", values="inf", aggfunc="mean") * 100)
print()
# churn do mensal em jun-jul/26 por tempo de casa e canal de aquisição: Google Play contra App Store
jj = m[m.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"])) & m.canal_cobranca.isin(["app_store", "google_play"])].merge(assin[["user_id", "inicio"]], on="user_id")
jj["casa"] = pd.cut((jj.mes - jj.inicio).dt.days, [-1, 90, 180, 365, 5000], labels=["até 3 meses", "3-6 meses", "6-12 meses", "12+ meses"])
print("jun-jul/26, mensal: churn % por tempo de casa"); x = churn(jj, ["casa", "canal_cobranca"]); print(x["churn %"].unstack().join(x.ativas.unstack(), rsuffix=" (ativas)"))
print(); print("jun-jul/26, mensal: churn % por canal de aquisição"); x = churn(jj, ["canal_aquisicao", "canal_cobranca"]); print(x["churn %"].unstack().join(x.ativas.unstack(), rsuffix=" (ativas)"))
print(); print("jun-jul/26, mensal: churn % por versão do app (na exportação)"); x = churn(jj, ["versao_app", "canal_cobranca"]); print(x["churn %"].unstack().join(x.ativas.unstack(), rsuffix=" (ativas)"))

preço cobrado no mensal (aprovado), por método: valor mais comum por mês
metodo      app_store  cartao  google_play  pix_automatico
mes                                                       
2026-01-01        0.0     0.0          0.0             0.0
2026-02-01        0.0     0.0          0.0             0.0
2026-03-01      100.0   100.0        100.0           100.0
2026-04-01      100.0   100.0        100.0           100.0
2026-05-01      100.0   100.0        100.0           100.0
2026-06-01      100.0   100.0        100.0           100.0
2026-07-01      100.0   100.0        100.0           100.0
2026-08-01      100.0   100.0        100.0           100.0
2026-09-01      100.0   100.0        100.0           100.0

novas assinaturas mensais por semestre e canal
canal_cobranca  app_store  google_play  web_cartao  web_pix
semestre                                                   
2025-S1              1266         1301         858      476
2025-S2              1249         1120         757

jun-jul/26, mensal: churn % por tempo de casa
canal_cobranca  app_store  google_play  app_store (ativas)  google_play (ativas)
casa                                                                            
até 3 meses          10.1         15.6                1158                  1037
3-6 meses            11.5         15.9                 958                   885
6-12 meses            7.8         12.1                 941                   750
12+ meses             5.4         10.3                1205                  1215

jun-jul/26, mensal: churn % por canal de aquisição
canal_cobranca        app_store  google_play  app_store (ativas)  google_play (ativas)
canal_aquisicao                                                                       
google_ads                  7.7         17.2                 417                   431
indicacao                   6.7          8.8                 600                   603
influenciador              13.1         16.6                 758    

C:\Users\Inteli\AppData\Local\Temp\ipykernel_9060\2365940271.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = d.groupby(por).agg(ativas=("ativa_inicio_mes", "sum"), perdas=("perdida_no_mes", "sum"))


## 3. Falha de GPS do app (04/05 a 28/06/2026)

O registro de decisões já marca treinos ao ar livre do app sem distância ou ritmo nesse período. Em que sistema aconteceu?

In [5]:
t = ativ.merge(usuarios[["user_id", "sistema"]], on="user_id", how="left")
ar = t[t.tipo.isin(["corrida", "caminhada", "pedal"]) & (t.origem == "app")]
print("treinos ao ar livre gravados no app:", len(ar))
f = ar[~ar.gps_ok]
print("com falha de GPS:", len(f), "| de", f.data.min().date(), "a", f.data.max().date(), "| contas:", f.user_id.nunique())
print(); print("falhas por sistema"); print(f.groupby("sistema").agg(treinos=("data", "size"), contas=("user_id", "nunique")))
jan = ar[(ar.data >= "2026-05-04") & (ar.data <= "2026-06-28")]
g = jan.groupby("sistema").agg(treinos=("data", "size"), falhas=("gps_ok", lambda s: (~s).sum()), contas=("user_id", "nunique"))
g["% com falha"] = g.falhas / g.treinos * 100
print(); print("na janela de 04/05 a 28/06/2026"); print(g)
print(); print("% dos treinos ao ar livre do app com falha, por semana e sistema")
sem = ar[(ar.data >= "2026-04-20") & (ar.data <= "2026-07-12")].assign(semana=lambda d: d.data.dt.to_period("W").dt.start_time)
print((sem.assign(falha=~sem.gps_ok).groupby(["semana", "sistema"]).falha.mean() * 100).unstack())

treinos ao ar livre gravados no app: 482020
com falha de GPS: 14778 | de 2026-05-04 a 2026-06-28 | contas: 4319

falhas por sistema
         treinos  contas
sistema                 
android    14778    4319

na janela de 04/05 a 28/06/2026
         treinos  falhas  contas  % com falha
sistema                                      
android    47279   14778    7153         31.3
ios        48516       0    6937          0.0

% dos treinos ao ar livre do app com falha, por semana e sistema
sistema     android  ios
semana                  
2026-04-20      0.0  0.0
2026-04-27      0.0  0.0
2026-05-04     32.1  0.0
2026-05-11     31.4  0.0
2026-05-18     31.4  0.0
2026-05-25     31.5  0.0
2026-06-01     31.5  0.0
2026-06-08     31.1  0.0
2026-06-15     30.8  0.0
2026-06-22     30.1  0.0
2026-06-29      0.0  0.0
2026-07-06      0.0  0.0


In [6]:
# Coorte: assinantes mensais ativos em 01/05/2026, acompanhados até o corte
base = painel[(painel.mes == "2026-05-01") & painel.ativa_inicio_mes & (painel.plano == "mensal")][["user_id", "canal_cobranca"]].merge(usuarios[["user_id", "sistema"]], on="user_id")
print("mensais ativos em 01/05/2026:", len(base))
fim = assin.set_index("user_id")[["data_fim", "motivo_fim", "ativa_no_corte"]]
base = base.join(fim, on="user_id")
# uso ao ar livre pelo app nos 60 dias antes da falha (05/03 a 03/05) e falhas na janela
antes = ar[(ar.data >= "2026-03-05") & (ar.data <= "2026-05-03")].groupby("user_id").size().rename("ar_livre_antes")
nf = f.groupby("user_id").size().rename("falhas")
nj = jan.groupby("user_id").size().rename("ar_livre_janela")
base = base.join(antes, on="user_id").join(nf, on="user_id").join(nj, on="user_id").fillna({"ar_livre_antes": 0, "falhas": 0, "ar_livre_janela": 0})
base["grupo"] = np.select([base.falhas > 0, base.ar_livre_janela > 0, base.ar_livre_antes > 0],
                          ["a. teve falha de GPS", "b. treinou ao ar livre no app, sem falha", "c. treinava ao ar livre antes, não na janela"], "d. não treina ao ar livre pelo app")
base["saiu_mai_ago"] = base.data_fim.between("2026-05-01", "2026-08-31")
base["saiu_jun_jul"] = base.data_fim.between("2026-06-01", "2026-07-31")
g = base.groupby(["sistema", "grupo"]).agg(assinantes=("user_id", "size"), saiu_jun_jul=("saiu_jun_jul", "sum"), saiu_mai_ago=("saiu_mai_ago", "sum"))
g["% saiu jun-jul"] = g.saiu_jun_jul / g.assinantes * 100
g["% saiu mai-ago"] = g.saiu_mai_ago / g.assinantes * 100
print(g)
print()
print("mesma leitura só para loja (Google Play × App Store)")
g2 = base[base.canal_cobranca.isin(["app_store", "google_play"])].groupby(["canal_cobranca", "grupo"]).agg(assinantes=("user_id", "size"), saiu_mai_ago=("saiu_mai_ago", "mean"))
g2["saiu_mai_ago"] *= 100; print(g2.unstack(0))
print()
base["n_falhas"] = pd.cut(base.falhas, [-1, 0, 1, 3, 100], labels=["0", "1", "2-3", "4+"])
x = base[(base.sistema == "android") & (base.ar_livre_janela > 0)].groupby("n_falhas").agg(assinantes=("user_id", "size"), saiu=("saiu_mai_ago", "mean"))
x["saiu"] *= 100; print("android que treinou ao ar livre no app na janela: % saiu mai-ago por nº de falhas"); print(x)
g.to_csv(S / "google_play_falha_gps_coorte.csv", encoding="utf-8-sig")

mensais ativos em 01/05/2026: 6149
                                                      assinantes  saiu_jun_jul  saiu_mai_ago  % saiu jun-jul  % saiu mai-ago
sistema grupo                                                                                                               
android a. teve falha de GPS                                1572           395           570            25.1            36.3
        b. treinou ao ar livre no app, sem falha             831           152           251            18.3            30.2
        c. treinava ao ar livre antes, não na janela         482            94           331            19.5            68.7
        d. não treina ao ar livre pelo app                   275            44           131            16.0            47.6
ios     b. treinou ao ar livre no app, sem falha            2420           333           596            13.8            24.6
        c. treinava ao ar livre antes, não na janela         421            59           2

android que treinou ao ar livre no app na janela: % saiu mai-ago por nº de falhas
          assinantes  saiu
n_falhas                  
0                831  30.2
1                538  41.8
2-3              461  36.4
4+               573  30.9


C:\Users\Inteli\AppData\Local\Temp\ipykernel_9060\3825390809.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  x = base[(base.sistema == "android") & (base.ar_livre_janela > 0)].groupby("n_falhas").agg(assinantes=("user_id", "size"), saiu=("saiu_mai_ago", "mean"))


In [7]:
# Tamanho do efeito em jun-jul/2026: perdas do mensal acima do que haveria com a taxa do iOS / da App Store
jj2 = m[m.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"]))]
g = churn(jj2, "sistema"); print(g)
esp = g.loc["android", "ativas"] * g.loc["ios", "churn %"] / 100
print("Android: perdas", int(g.loc["android", "perdas"]), "| esperadas com a taxa do iOS:", round(esp), "| a mais:", round(g.loc["android", "perdas"] - esp))
g = churn(jj2, "canal_cobranca"); print(g)
esp = g.loc["google_play", "ativas"] * g.loc["app_store", "churn %"] / 100
print("Google Play: perdas", int(g.loc["google_play", "perdas"]), "| esperadas com a taxa da App Store:", round(esp), "| a mais:", round(g.loc["google_play", "perdas"] - esp))
print()
# Comparação limpa: só quem treinou ao ar livre pelo app na janela da falha (mesmo hábito, sistemas diferentes)
k = base[base.ar_livre_janela > 0].assign(g=lambda d: np.where(d.sistema == "ios", "ios", np.where(d.falhas > 0, "android com falha registrada", "android sem falha registrada")))
x = k.groupby("g").agg(assinantes=("user_id", "size"), saiu_jun_jul=("saiu_jun_jul", "mean"), saiu_mai_ago=("saiu_mai_ago", "mean"))
x[["saiu_jun_jul", "saiu_mai_ago"]] *= 100; print(x)
print()
# 2026 sem jun-jul: o Android ainda sai mais?
r = churn(m[(m.ano == 2026) & ~m.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"]))], "sistema"); print("mensal, 2026 sem jun-jul"); print(r)
r = churn(m[(m.ano == 2026) & ~m.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"]))], "canal_cobranca"); print(r)

         ativas  perdas  churn %
sistema                         
android    5963     843     14.1
ios        5872     505      8.6
Android: perdas 843 | esperadas com a taxa do iOS: 513 | a mais: 330
                ativas  perdas  churn %
canal_cobranca                         
app_store         4262     365      8.6
google_play       3887     519     13.4
web_cartao        2267     286     12.6
web_pix           1419     178     12.5
Google Play: perdas 519 | esperadas com a taxa da App Store: 333 | a mais: 186

                              assinantes  saiu_jun_jul  saiu_mai_ago
g                                                                   
android com falha registrada        1572          25.1          36.3
android sem falha registrada         831          18.3          30.2
ios                                 2420          13.8          24.6

mensal, 2026 sem jun-jul
         ativas  perdas  churn %
sistema                         
android   21922    2434     11.1
ios      

## 4. Suporte e motivo declarado

In [8]:
tk = tickets.merge(usuarios[["user_id", "sistema"]], on="user_id", how="left")
print("chamados:", len(tickets), "| sem conta em usuarios_limpa:", int(tk.sistema.isna().sum()), "| sobre falha de GPS:", int(tk.sobre_falha_gps.sum()))
tk["mes"] = tk.aberto_em.dt.to_period("M")
print("chamados sobre falha de GPS por mês e sistema"); print(pd.crosstab(tk[tk.sobre_falha_gps].mes, tk[tk.sobre_falha_gps].sistema))
print(); print("chamados técnicos (categoria do texto) por mês e sistema");
k = tk[(tk.categoria_do_texto == "tecnico") & (tk.aberto_em >= "2026-01-01")]; print(pd.crosstab(k.mes, k.sistema))
print(); print("status no corte dos chamados sobre GPS"); print(tk[tk.sobre_falha_gps].status_no_corte.value_counts().to_dict())
print(); print("textos mais comuns sobre GPS"); print(tk[tk.sobre_falha_gps].texto_cliente.value_counts().head(6).to_string())
print()
ps = pesq[pesq.resposta_mais_recente].merge(usuarios[["user_id", "sistema"]], on="user_id", how="left").merge(assin[["user_id", "canal_cobranca"]], on="user_id", how="left")
print("pesquisa de saída (só cancelamento pelo site, jun-set/2026):", len(ps))
print(pd.crosstab(ps.motivo_ajustado, ps.sistema, margins=True)); print((pd.crosstab(ps.motivo_ajustado, ps.sistema, normalize="columns") * 100))

chamados: 4422 | sem conta em usuarios_limpa: 0 | sobre falha de GPS: 1191
chamados sobre falha de GPS por mês e sistema
sistema  android
mes             
2026-05      675
2026-06      516

chamados técnicos (categoria do texto) por mês e sistema
sistema  android  ios
mes                  
2026-01        6    4
2026-02        3    2
2026-03        5    8
2026-04        5    4
2026-05      667    2
2026-06      504    3
2026-07        5   10
2026-08        6    5
2026-09       10   10

status no corte dos chamados sobre GPS
{'aberto': 532, 'resolvido': 367, 'sem_resposta': 292}

textos mais comuns sobre GPS
texto_cliente
corrida registrada com distância zerada               224
perdi meus treinos da semana, o app travou no meio    212
a rota sai toda torta e a distância errada            199
o app não grava meu percurso desde a atualização      183
GPS parou de funcionar no meu android                 154
gps parou de funcionar no meu android                  88



pesquisa de saída (só cancelamento pelo site, jun-set/2026): 641
sistema            android  ios  All
motivo_ajustado                     
nao_uso                 49   27   76
objetivo_cumprido       91   65  156
outro                   89   49  138
outro_app               16   15   31
preco                  128   71  199
problema_tecnico        26    1   27
saude                    8    6   14
All                    407  234  641
sistema            android  ios
motivo_ajustado                
nao_uso               12.0 11.5
objetivo_cumprido     22.4 27.8
outro                 21.9 20.9
outro_app              3.9  6.4
preco                 31.4 30.3
problema_tecnico       6.4  0.4
saude                  2.0  2.6


## 5. Engajamento antes da saída: Google Play contra App Store

In [9]:
s = assin[(assin.plano_atual == "mensal") & assin.canal_cobranca.isin(["app_store", "google_play"]) & (assin.data_fim >= "2026-03-17") & (assin.motivo_fim == "renovacao_desligada")].copy()
print("saídas voluntárias do mensal nas lojas, 17/03 a 30/09/2026:", len(s))
e = niveis_engajamento(ativ, s.user_id, s.set_index("user_id").data_fim - pd.Timedelta(days=1), inicio=s.set_index("user_id").inicio)
s = s.merge(e[["user_id", "nivel", "treinos_30d"]], on="user_id")
s["periodo"] = np.where(s.data_fim.between("2026-06-01", "2026-07-31"), "jun-jul/26", "outros meses")
n = pd.crosstab([s.periodo, s.canal_cobranca], s.nivel).reindex(columns=NIVEIS)
print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("saídas")).join(s.groupby(["periodo", "canal_cobranca"]).treinos_30d.mean().rename("treinos 30d (média)")))
print()
eng = pd.read_parquet(P / "engajamento_corte.parquet").merge(assin[["user_id", "canal_cobranca", "plano_atual"]], on="user_id")
k = eng[(eng.plano_atual == "mensal") & eng.canal_cobranca.isin(["app_store", "google_play"])]
n = pd.crosstab(k.canal_cobranca, k.nivel).reindex(columns=NIVEIS)
print("ativos no corte, mensal"); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")))

saídas voluntárias do mensal nas lojas, 17/03 a 30/09/2026: 2730


                             1. alto  2. médio  3. baixo  4. possível churn  saídas  treinos 30d (média)
periodo      canal_cobranca                                                                             
jun-jul/26   app_store          27.5      57.9      11.7                2.9     342                  4.6
             google_play        27.8      56.5      13.1                2.6     504                  4.3
outros meses app_store          31.0      55.3      10.4                3.3     944                  4.5
             google_play        30.4      55.3      11.1                3.2     940                  4.4

ativos no corte, mensal
                1. alto  2. médio  3. baixo  4. possível churn  ativos
canal_cobranca                                                        
app_store          86.9      12.6       0.4                0.1    2348
google_play        86.5      12.9       0.5                0.1    2036
